# Company Brochure Generator

## Business Challenge

Build a tool that **automatically generates a professional company brochure** given just a company name and website URL.

The brochure should be suitable for:
- **Prospective clients**: understand what the company does
- **Investors**: learn about the company's mission and growth
- **Recruits**: discover company culture and open positions

### How it Works

1. **Scrape** the company's website for content and links
2. Use an LLM to **identify relevant links** (About, Careers, etc.)
3. **Scrape those relevant pages** for additional context
4. Use an LLM to **generate a polished brochure** in markdown

### Key Concepts
- Multi-step LLM pipelines (early Agentic AI patterns)
- One-shot prompting with JSON structured output
- Web scraping with Scrapling
- Streaming responses for better UX

---
## Setup & Imports

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from my_scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODEL = 'gemini-2.5-flash'

gemini = OpenAI(api_key=api_key, base_url=GEMINI_BASE_URL)

---
## Step 1: Scrape the Website for Links

First, we fetch all the links on a company's landing page.  
We'll then use an LLM to decide which links are relevant for a brochure.

In [6]:
links = fetch_website_links("https://huggingface.co")
links

[2026-03-03 17:36:14] INFO: Fetched (200) <GET https://huggingface.co/> (referer: https://www.google.com/search?q=huggingface)


['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Qwen/Qwen3.5-35B-A3B',
 '/Qwen/Qwen3.5-27B',
 '/unsloth/Qwen3.5-35B-A3B-GGUF',
 '/Qwen/Qwen3.5-122B-A10B',
 '/Qwen/Qwen3.5-9B',
 '/models',
 '/spaces/multimodalart/qwen-image-multiple-angles-3d-camera',
 '/spaces/FrameAI4687/Omni-Video-Factory',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/mrfakename/Z-Image-Turbo',
 '/spaces/Qwen/Qwen3-TTS',
 '/spaces',
 '/datasets/peteromallet/dataclaw-peteromallet',
 '/datasets/togethercomputer/CoderForge-Preview',
 '/datasets/nohurry/Opus-4.6-Reasoning-3000x-filtered',
 '/datasets/crownelius/Opus-4.6-Reasoning-3300x',
 '/datasets/nvidia/Nemotron-Terminal-Corpus',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/facebook',
 '/amazon',


---
## Step 2: Use the LLM to Select Relevant Links

We use **one-shot prompting** providing an example of the expected JSON response format  
so the model knows exactly how to structure its output.

This is a great use case for an LLM because it requires *nuanced judgement*  
(e.g., an "About" page is relevant, but "Terms of Service" is not).

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_links_user_prompt(url):
    """Build a prompt with all the links from a website for the LLM to evaluate."""
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
def select_relevant_links(url):
    """Use the LLM to pick the most relevant links from a website."""
    print(f"Selecting relevant links    for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash


[2026-03-03 17:36:33] INFO: Fetched (200) <GET https://huggingface.co/> (referer: https://www.google.com/search?q=huggingface)


Found 9 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'company profile page',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'}]}

---
## Step 3: Gather All Relevant Content

Now we scrape both the landing page **and** each relevant link the LLM identified.  
This gives the brochure generator much richer context to work with.

In [6]:
def fetch_page_and_all_relevant_links(url):
    """Scrape the landing page + all LLM-selected relevant pages."""
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        try:
            result += fetch_website_contents(link["url"])
        except Exception as e:
            result += f"(Could not fetch: {e})"
    return result

---
## Step 4: Generate the Brochure

We assemble everything into a final prompt and ask the LLM to create a polished brochure.

In [7]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [8]:
def get_brochure_user_prompt(company_name, url):
    """Build the final prompt with all gathered content."""
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt

In [9]:
def create_brochure(company_name, url):
    """Generate and display a company brochure."""
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [10]:
create_brochure("Hugging Face", "https://huggingface.co")

[2026-03-03 17:50:59] INFO: Fetched (200) <GET https://huggingface.co/> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:50:59] INFO: Fetched (200) <GET https://huggingface.co/> (referer: https://www.google.com/search?q=huggingface)


Selecting relevant links    for https://huggingface.co by calling gemini-2.5-flash


[2026-03-03 17:51:10] INFO: Fetched (200) <GET https://huggingface.co/> (referer: https://www.google.com/search?q=huggingface)


Found 18 relevant links


[2026-03-03 17:51:11] INFO: Fetched (200) <GET https://huggingface.co/models> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:51:12] INFO: Fetched (200) <GET https://huggingface.co/datasets> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:51:13] INFO: Fetched (200) <GET https://huggingface.co/spaces> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:51:13] INFO: Fetched (200) <GET https://huggingface.co/docs> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:51:14] INFO: Fetched (200) <GET https://huggingface.co/enterprise> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:51:15] INFO: Fetched (200) <GET https://huggingface.co/pricing> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:51:16] INFO: Fetched (200) <GET https://huggingface.co/inference/models> (referer: https://www.google.com/search?q=huggingface)
[2026-03-03 17:51:17] INFO: Fetched (200) <GET https:/

**Hugging Face: The AI Community Building the Future**

Welcome to Hugging Face, the central hub for the machine learning community. We are dedicated to accelerating the advancement of AI by providing a collaborative platform where innovators worldwide can create, discover, and build the future of machine learning.

**What We Offer:**

Hugging Face is the premier platform for hosting and collaborating on a vast ecosystem of machine learning resources. Our platform empowers developers, researchers, and organizations to:

*   **Explore & Contribute:** Access over 2 million models, 1 million applications, and 500,000 datasets across diverse modalities including text, image, video, audio, and even 3D.
*   **Collaborate Seamlessly:** Leverage our collaboration platform to host unlimited public models, datasets, and applications, fostering shared progress and innovation.
*   **Move Faster:** Utilize the robust Hugging Face open-source stack and integrate with leading ML libraries like PyTorch, TensorFlow, JAX, Transformers, and Diffusers.
*   **Build Your Portfolio:** Share your work with a global audience and establish your reputation within the ML community.

We support a wide array of ML tasks, from advanced text generation, image-to-text, and text-to-image, to video generation and custom speech synthesis.

**For Our Customers:**

*   **Individuals & Researchers:** Unleash your potential with unparalleled access to cutting-edge models and datasets, a powerful platform to develop and share your AI projects, and a vibrant community to learn from and contribute to.
*   **Teams & Enterprises:** Accelerate your ML initiatives with our paid Compute and Enterprise solutions. Gain access to advanced platforms designed to boost productivity, streamline workflows, and ensure your team has the resources to build the most sophisticated AI applications.

**Our Culture:**

At our core, Hugging Face is about **community and collaboration**. We believe in the power of open source and collective intelligence to drive innovation. We foster an environment where sharing knowledge, tools, and breakthroughs is paramount to building the AI of tomorrow, together.

Join Hugging Face and be part of the community that's building the future of AI.

**Sign Up | Explore AI Apps | Browse Models | Team & Enterprise Solutions**

---
## Streaming Version

Stream the brochure as it's generated for a smooth typewriter effect.

In [16]:
def stream_brochure(company_name, url):
    """Generate and stream a company brochure with a typewriter effect."""
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [17]:
stream_brochure("Anthropic", "https://anthropic.com")

[2026-03-03 17:40:04] INFO: Fetched (200) <GET https://www.anthropic.com/> (referer: https://www.google.com/search?q=anthropic)


Selecting relevant links for https://anthropic.com by calling gemini-2.5-flash


[2026-03-03 17:40:05] INFO: Fetched (200) <GET https://www.anthropic.com/> (referer: https://www.google.com/search?q=anthropic)


Found 12 relevant links


[2026-03-03 17:41:18] INFO: Fetched (200) <GET https://www.anthropic.com/company> (referer: https://www.google.com/search?q=anthropic)
[2026-03-03 17:41:20] INFO: Fetched (200) <GET https://www.anthropic.com/careers> (referer: https://www.google.com/search?q=anthropic)
[2026-03-03 17:41:24] INFO: Fetched (200) <GET https://www.anthropic.com/research> (referer: https://www.google.com/search?q=anthropic)
[2026-03-03 17:41:26] INFO: Fetched (200) <GET https://www.anthropic.com/economic-futures> (referer: https://www.google.com/search?q=anthropic)
[2026-03-03 17:41:30] INFO: Fetched (200) <GET https://www.anthropic.com/constitution> (referer: https://www.google.com/search?q=anthropic)
[2026-03-03 17:41:34] INFO: Fetched (200) <GET https://www.anthropic.com/transparency> (referer: https://www.google.com/search?q=anthropic)
[2026-03-03 17:41:35] INFO: Fetched (200) <GET https://www.anthropic.com/responsible-scaling-policy> (referer: https://www.google.com/search?q=anthropic)
[2026-03-03 17:4

## Anthropic: Building Trustworthy AI for a Better Future

Anthropic is a public benefit corporation at the forefront of AI research and development, dedicated to ensuring artificial intelligence serves the global good. We believe AI will have a vast and transformative impact on the world, and our mission is to secure its profound benefits while diligently mitigating its inherent risks.

**Our Vision & Approach**
As an AI safety and research company, we are committed to building frontier AI systems that are reliable, interpretable, and steerable. We treat AI safety as a systematic science, conducting advanced research, developing robust safety techniques, and integrating these into all our products. This rigorous approach allows us to create AI that people can truly rely on. Our foundational principles guide us to act for the global good, fostering an environment where powerful AI systems go well for everyone.

**Innovative Products**
At the heart of our offerings is **Claude**, our flagship AI assistant designed to be helpful, honest, and harmless. We empower users and developers through:
*   **Claude:** Our leading conversational AI.
*   **Claude Code:** Specialized for coding tasks.
*   **Claude Developer Platform:** Providing tools and APIs for integration.
*   **Advanced Models:** Including Opus, Sonnet, and Haiku, offering a spectrum of capabilities.

**For Our Customers & Partners**
Anthropic provides cutting-edge AI solutions built with an unparalleled focus on safety, transparency, and trust. From advanced applications like AI-planned planetary drives to everyday use cases, our products are engineered for dependability and responsible deployment. We offer robust security and compliance, detailed transparency reports, and a responsible scaling policy to ensure peace of mind.

**For Investors & Stakeholders**
Investing in Anthropic means investing in the responsible future of AI. As a public benefit corporation, our commitment to safety and ethical development is core to our long-term value and sustainability. Our extensive research into economic futures, commitments, and initiatives, including Claude's Constitution, demonstrates our dedication to shaping a beneficial AI landscape.

**Join Our Team: Careers at Anthropic**
Are you drawn to hard problems with real stakes? At Anthropic, we are a diverse team of researchers, engineers, and builders working to shape how AI meets the world. We offer a unique opportunity to contribute to pioneering AI safety research and develop systems that will positively impact humanity. If you are passionate about making powerful AI go well for everyone and thrive in a challenging, mission-driven environment, we invite you to explore our open roles and join us in building the future of reliable AI.

Discover more at Anthropic and experience AI you can rely on.

---
## Try Your Own

Replace the company name and URL below to generate brochures for other companies.

In [ ]:
# Try it with another company
stream_brochure("Anthropic", "https://anthropic.com")